# 实验四：从零实现字符级 Vanilla RNN

欢迎来到实验四的补充实验。本 Notebook 基于 `minimalRNN.py`（Andrej Karpathy 的极简字符级 RNN）和博客 [The Unreasonable Effectiveness of Recurrent Neural Networks](https://karpathy.github.io/2015/05/21/rnn-effectiveness/) 改写而来。你将使用 **NumPy 从零搭建一个 Vanilla RNN 字符级语言模型**，并完整走通数据准备、前向传播、通过时间反向传播（BPTT）、梯度裁剪、AdaGrad 训练和采样生成。

原始 `minimalRNN.py` 很短，但包含 RNN 语言模型的完整闭环：

1. 把文本拆成字符词表，并将字符映射为整数 token。
2. 用 one-hot 向量作为每个时间步的输入。
3. 用隐藏状态 $h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b_h)$ 保存历史信息。
4. 用输出层 $y_t = W_{hy}h_t + b_y$ 预测下一个字符。
5. 用 softmax + 交叉熵计算语言模型损失。
6. 手写 BPTT，沿时间反向累积梯度。
7. 对梯度做裁剪，缓解梯度爆炸。
8. 用 AdaGrad 更新参数，并定期采样生成文本。

---

**约定：** 本 Notebook 中凡是出现 `### START CODE HERE ###` 与 `### END CODE HERE ###` 之间的区域，均需要学生手动补全代码。当前提交的是教师答案版，因此这些区域已经填好；发布脚本会自动清空这些区域中的答案。

> 实验环境：Python 3.10+，NumPy。无需 PyTorch。为了保证单文件可运行，本实验内嵌一个小型字符语料，不依赖外部 `input.txt`。

## 0. 博客与代码的对应关系

Karpathy 的博客先解释“为什么 RNN 适合序列”：RNN 的输出不仅取决于当前输入，还取决于过去输入累积成的隐藏状态。随后博客用字符级语言模型落地这个思想：给模型一大段文本，让它在每个时间步预测下一个字符，然后在测试时把采样出的字符再喂回模型，循环生成文本。

`minimalRNN.py` 正是博客中“最小字符级 RNN”的 NumPy 版本。本实验做的是把这份约 100 行代码拆成可学习的模块：

| 博客中的思想 | `minimalRNN.py` 中的代码 | 本实验中的练习 |
|---|---|---|
| 字符级建模 | `chars`, `char_to_ix`, `ix_to_char` | 练习 1 |
| 1-of-k / one-hot 输入 | `xs[t] = np.zeros(...)` | 练习 3 |
| RNN 单步递推 | `hs[t] = np.tanh(...)` | 练习 4 |
| softmax 分类器与交叉熵 | `ps[t]`, `loss += -np.log(...)` | 练习 3、4 |
| 递归链式法则 / BPTT | 反向遍历 `reversed(...)` | 练习 5 |
| 稳定训练 | `np.clip`, AdaGrad | 练习 6、8 |
| 采样文本 | `sample(...)` | 练习 7、10 |
| 温度影响采样多样性 | 博客中的 temperature 讨论 | 选做思考 |

原始文件是 Python 2 写法，并使用 `while True` 无限训练。本实验保留变量名和算法结构，但改为 Python 3、有限步训练、内嵌小语料，方便课堂运行和自动测试。

In [1]:
import numpy as np

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

print("NumPy version:", np.__version__)

NumPy version: 1.26.4


## 1. 数据与字符级语言建模

字符级语言模型把文本看成一个很长的字符序列。给定当前位置之前的字符，模型要预测下一个字符。

例如序列 `anna\n` 会形成训练对：

| 输入字符 | 目标字符 |
|---|---|
| `a` | `n` |
| `n` | `n` |
| `n` | `a` |
| `a` | 换行符 |

博客用 `hello` 解释同一件事：第二个 `l` 和第一个 `l` 的当前字符相同，但目标字符不同，因此模型必须依靠隐藏状态保存上下文，而不能只看当前输入字符。

为了让 Notebook 自包含，下面直接内嵌一个小型名字语料，并重复多次形成训练文本。

In [2]:
raw_corpus = """
anna
anne
annie
abel
belle
bella
ben
benny
cara
carl
dora
doris
ella
ellen
emily
frank
grace
harry
helen
irene
julia
kevin
lily
maria
nora
oliver
paul
quincy
rachel
simon
tina
ursula
victor
wendy
xavier
yara
zoe
""".strip().lower()

# 重复语料是为了让一个很小的 RNN 在短时间内也能看到足够多的字符。
data = (raw_corpus + "\n") * 35

print("语料字符数:", len(data))
print("语料预览:")
print(data[:160])

语料字符数: 7420
语料预览:
anna
anne
annie
abel
belle
bella
ben
benny
cara
carl
dora
doris
ella
ellen
emily
frank
grace
harry
helen
irene
julia
kevin
lily
maria
nora
oliver
paul
quincy
ra


### 练习 1：构建字符词表与索引映射

你需要完成 `minimalRNN.py` 开头的数据 I/O 部分：

```python
chars = list(set(data))
data_size, vocab_size = len(data), len(chars)
char_to_ix = { ch:i for i,ch in enumerate(chars) }
ix_to_char = { i:ch for i,ch in enumerate(chars) }
```

本实验使用 `sorted(set(data))` 而不是 `list(set(data))`，这样每次运行时字符顺序固定，测试结果也固定。

需要得到：

- `chars`：所有唯一字符组成的列表；
- `data_size`：语料总字符数；
- `vocab_size`：词表大小；
- `char_to_ix`：字符到整数 id 的映射；
- `ix_to_char`：整数 id 到字符的映射。

In [3]:
### START CODE HERE ###
# 1. 收集唯一字符。为保证可复现，使用 sorted 固定顺序。

# 2. 记录数据规模和词表规模。

# 3. 建立双向映射：字符 -> id，id -> 字符。
### END CODE HERE ###

print(f"data has {data_size} characters, {vocab_size} unique.")
print("chars:", repr(''.join(chars)))
print("'a' ->", char_to_ix['a'], "->", ix_to_char[char_to_ix['a']])

data has 7420 characters, 27 unique.
chars: '\nabcdefghijklmnopqrstuvwxyz'
'a' -> 1 -> a


In [4]:
# ===== 测试单元格（无需修改）=====
def test_tokenizer():
    expected_chars = sorted(set(data))
    assert chars == expected_chars, "chars 应等于 sorted(set(data))"
    assert data_size == len(data), "data_size 应等于 len(data)"
    assert vocab_size == len(expected_chars), "vocab_size 应等于唯一字符数"
    assert set(char_to_ix.keys()) == set(chars), "char_to_ix 的键应覆盖全部字符"
    assert set(ix_to_char.keys()) == set(range(vocab_size)), "ix_to_char 的键应为 0..vocab_size-1"
    for ch in chars:
        assert ix_to_char[char_to_ix[ch]] == ch, f"字符 {repr(ch)} 的双向映射不一致"
    assert '\n' in char_to_ix, "字符级建模需要保留换行符作为普通字符"
    print("✅ 练习 1 通过：词表和双向映射正确。")

test_tokenizer()

✅ 练习 1 通过：词表和双向映射正确。


## 2. 模型参数

Vanilla RNN 在每个时间步使用同一组参数：

$$h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b_h)$$

$$y_t = W_{hy}h_t + b_y$$

其中：

| 参数 | 含义 | 形状 |
|---|---|---|
| `Wxh` | input-to-hidden | `(hidden_size, vocab_size)` |
| `Whh` | hidden-to-hidden | `(hidden_size, hidden_size)` |
| `Why` | hidden-to-output | `(vocab_size, hidden_size)` |
| `bh` | hidden bias | `(hidden_size, 1)` |
| `by` | output bias | `(vocab_size, 1)` |

`minimalRNN.py` 使用小随机数初始化权重、零初始化偏置。

### 练习 2：初始化 RNN 参数

补全权重和偏置初始化。注意矩阵形状应与上表一致。

In [5]:
# 超参数。原始 minimalRNN.py 使用 hidden_size=100；这里稍小一些，方便课堂快速运行。
hidden_size = 50
seq_length = 25
learning_rate = 1e-1

np.random.seed(1)

### START CODE HERE ###
# 输入到隐藏层：one-hot 字符向量 -> 隐藏状态

# 上一隐藏状态到当前隐藏状态

# 隐藏状态到输出 logits

# 两个偏置项
### END CODE HERE ###

parameters = {"Wxh": Wxh, "Whh": Whh, "Why": Why, "bh": bh, "by": by}

print("参数形状:")
for name, value in parameters.items():
    print(f"  {name}: {value.shape}")

参数形状:
  Wxh: (50, 27)
  Whh: (50, 50)
  Why: (27, 50)
  bh: (50, 1)
  by: (27, 1)


In [6]:
# ===== 测试单元格（无需修改）=====
def test_parameter_shapes():
    assert Wxh.shape == (hidden_size, vocab_size), "Wxh 形状错误"
    assert Whh.shape == (hidden_size, hidden_size), "Whh 形状错误"
    assert Why.shape == (vocab_size, hidden_size), "Why 形状错误"
    assert bh.shape == (hidden_size, 1), "bh 形状错误"
    assert by.shape == (vocab_size, 1), "by 形状错误"
    assert np.allclose(bh, 0), "bh 应初始化为 0"
    assert np.allclose(by, 0), "by 应初始化为 0"
    assert np.std(Wxh) > 0 and np.std(Whh) > 0 and np.std(Why) > 0, "权重不应全为 0"
    assert max(abs(Wxh).max(), abs(Whh).max(), abs(Why).max()) < 0.1, "权重应为小随机数"
    print("✅ 练习 2 通过：参数形状与初始化正确。")

test_parameter_shapes()

✅ 练习 2 通过：参数形状与初始化正确。


## 3. One-hot 与 Softmax

`minimalRNN.py` 每个时间步都把字符 id 编码成 one-hot 列向量：

```python
xs[t] = np.zeros((vocab_size, 1))
xs[t][inputs[t]] = 1
```

输出层得到的是未归一化的 logits。要把 logits 转成概率分布，需要 softmax：

$$p_i = \frac{e^{y_i}}{\sum_j e^{y_j}}$$

工程上应先减去最大 logit 再指数化：

$$p_i = \frac{e^{y_i - \max_j y_j}}{\sum_j e^{y_j - \max_j y_j}}$$

这不会改变 softmax 结果，但能避免 `np.exp` 溢出。

### 练习 3：实现 `one_hot` 与稳定 `softmax`

补全两个基础函数。后续前向传播、采样和训练都会依赖它们。

In [7]:
### START CODE HERE ###
    # 创建形状为 (size, 1) 的零列向量。
    # 将第 ix 个位置置为 1。


    # logits 是形状 (vocab_size, 1) 的列向量。
    # 先减去最大值，提升数值稳定性。
### END CODE HERE ###

print("one_hot(3, 8).T =", one_hot(3, 8).T)
print("softmax([[1], [2], [3]]).T =", softmax(np.array([[1.0], [2.0], [3.0]])).T)

one_hot(3, 8).T = [[0. 0. 0. 1. 0. 0. 0. 0.]]
softmax([[1], [2], [3]]).T = [[0.09   0.2447 0.6652]]


In [8]:
# ===== 测试单元格（无需修改）=====
def test_one_hot_and_softmax():
    x = one_hot(2, 5)
    assert x.shape == (5, 1), "one_hot 输出应为列向量"
    assert x.sum() == 1 and x[2, 0] == 1, "one_hot 只有目标位置为 1"

    logits = np.array([[1000.0], [1001.0], [999.0]])
    probs = softmax(logits)
    assert probs.shape == logits.shape, "softmax 应保持输入形状"
    assert np.all(np.isfinite(probs)), "softmax 不应产生 inf 或 nan"
    assert np.allclose(probs.sum(), 1.0), "softmax 概率和应为 1"

    expected = np.exp(np.array([[-1.0], [0.0], [-2.0]]))
    expected = expected / expected.sum()
    assert np.allclose(probs, expected), "softmax 数值不正确"
    print("✅ 练习 3 通过：one-hot 与稳定 softmax 正确。")

test_one_hot_and_softmax()

✅ 练习 3 通过：one-hot 与稳定 softmax 正确。


## 4. 前向传播：展开一个字符序列

给定整数列表 `inputs` 和 `targets`：

- `inputs[t]` 是第 $t$ 个输入字符 id；
- `targets[t]` 是第 $t$ 个位置要预测的下一个字符 id；
- `hprev` 是进入这段序列前的隐藏状态。

前向传播要保存每个时间步的中间变量，供反向传播使用：

| 缓存 | 内容 |
|---|---|
| `xs[t]` | one-hot 输入 $x_t$ |
| `hs[t]` | 隐藏状态 $h_t$ |
| `ys[t]` | logits $y_t$ |
| `ps[t]` | softmax 概率 $p_t$ |

总损失是每个时间步的负对数似然之和：

$$\mathcal{L} = \sum_t -\log p_t[\text{target}_t]$$

### 练习 4：实现 RNN 前向传播

按照公式补全 `rnn_forward`。注意：`hs[-1]` 存放进入序列前的隐藏状态 `hprev`，因此计算第 0 步时可以统一写成 `hs[t-1]`。

In [9]:
def rnn_forward(inputs, targets, hprev, parameters):
    Wxh, Whh, Why = parameters["Wxh"], parameters["Whh"], parameters["Why"]
    bh, by = parameters["bh"], parameters["by"]

    xs, hs, ys, ps = {}, {}, {}, {}
    hs[-1] = np.copy(hprev)
    loss = 0.0

    ### START CODE HERE ###
        # 1. 将字符 id 编码为 one-hot 列向量。

        # 2. 根据当前输入和上一隐藏状态计算当前隐藏状态。

        # 3. 计算输出 logits。

        # 4. softmax 得到下一个字符概率。

        # 5. 累加交叉熵损失。
    ### END CODE HERE ###

    cache = (xs, hs, ys, ps)
    hlast = hs[len(inputs) - 1]
    return loss, cache, hlast

print("rnn_forward 定义完成。")

rnn_forward 定义完成。


In [10]:
# ===== 测试单元格（无需修改）=====
def test_rnn_forward():
    sample_text = data[:seq_length + 1]
    inputs = [char_to_ix[ch] for ch in sample_text[:-1]]
    targets = [char_to_ix[ch] for ch in sample_text[1:]]
    h0 = np.zeros((hidden_size, 1))

    loss, cache, hlast = rnn_forward(inputs, targets, h0, parameters)
    xs, hs, ys, ps = cache

    assert np.isfinite(loss) and loss > 0, "loss 应为正的有限数"
    assert hlast.shape == (hidden_size, 1), "最后隐藏状态形状错误"
    assert len(xs) == len(inputs) and len(ps) == len(inputs), "缓存长度错误"
    for t in range(len(inputs)):
        assert xs[t].shape == (vocab_size, 1), f"xs[{t}] 形状错误"
        assert hs[t].shape == (hidden_size, 1), f"hs[{t}] 形状错误"
        assert ys[t].shape == (vocab_size, 1), f"ys[{t}] 形状错误"
        assert ps[t].shape == (vocab_size, 1), f"ps[{t}] 形状错误"
        assert np.allclose(ps[t].sum(), 1.0), f"ps[{t}] 不是概率分布"
    print(f"✅ 练习 4 通过：前向传播正确，loss = {loss:.4f}")

test_rnn_forward()

✅ 练习 4 通过：前向传播正确，loss = 82.3913


## 5. 反向传播：BPTT

RNN 的反向传播要从最后一个时间步往前走，因为当前隐藏状态会影响后续隐藏状态。

`minimalRNN.py` 中的核心反向公式如下：

```python
dy = np.copy(ps[t])
dy[targets[t]] -= 1

dWhy += np.dot(dy, hs[t].T)
dby += dy

dh = np.dot(Why.T, dy) + dhnext
dhraw = (1 - hs[t] * hs[t]) * dh

dbh += dhraw
dWxh += np.dot(dhraw, xs[t].T)
dWhh += np.dot(dhraw, hs[t-1].T)
dhnext = np.dot(Whh.T, dhraw)
```

其中 `dhnext` 表示从未来时间步传回当前隐藏状态的梯度。博客中说反向传播本质上是链式法则的递归应用；在 RNN 里，这个递归同时沿着网络层和时间轴展开。

### 练习 5：实现通过时间反向传播

补全 `rnn_backward`。这里先不做梯度裁剪，裁剪会在下一个练习中加入。

In [11]:
def rnn_backward(inputs, targets, parameters, cache):
    Wxh, Whh, Why = parameters["Wxh"], parameters["Whh"], parameters["Why"]
    xs, hs, ys, ps = cache

    dWxh = np.zeros_like(Wxh)
    dWhh = np.zeros_like(Whh)
    dWhy = np.zeros_like(Why)
    dbh = np.zeros_like(parameters["bh"])
    dby = np.zeros_like(parameters["by"])
    dhnext = np.zeros_like(hs[0])

    ### START CODE HERE ###
        # 1. softmax + cross-entropy 对 logits 的梯度。

        # 2. 输出层参数梯度。

        # 3. 当前隐藏状态收到两部分梯度：输出层梯度 + 未来时间步梯度。

        # 4. tanh'(z) = 1 - tanh(z)^2。这里 hs[t] 已经是 tanh 后的值。

        # 5. 输入到隐藏层、隐藏到隐藏层、隐藏偏置的梯度。

        # 6. 传给前一个时间步的隐藏状态梯度。
    ### END CODE HERE ###

    grads = {"dWxh": dWxh, "dWhh": dWhh, "dWhy": dWhy, "dbh": dbh, "dby": dby}
    return grads

print("rnn_backward 定义完成。")

rnn_backward 定义完成。


In [12]:
# ===== 测试单元格（无需修改）=====
def _copy_parameters(params):
    return {name: value.copy() for name, value in params.items()}


def _loss_for_params(test_params, inputs, targets, h0):
    loss, _, _ = rnn_forward(inputs, targets, h0, test_params)
    return loss


def test_rnn_backward():
    rng = np.random.default_rng(123)
    test_params = {
        "Wxh": rng.normal(0, 0.01, size=(hidden_size, vocab_size)),
        "Whh": rng.normal(0, 0.01, size=(hidden_size, hidden_size)),
        "Why": rng.normal(0, 0.01, size=(vocab_size, hidden_size)),
        "bh": np.zeros((hidden_size, 1)),
        "by": np.zeros((vocab_size, 1)),
    }
    inputs = [char_to_ix[ch] for ch in data[:5]]
    targets = [char_to_ix[ch] for ch in data[1:6]]
    h0 = np.zeros((hidden_size, 1))

    loss, cache, _ = rnn_forward(inputs, targets, h0, test_params)
    grads = rnn_backward(inputs, targets, test_params, cache)

    for name, param in test_params.items():
        grad = grads["d" + name]
        assert grad.shape == param.shape, f"d{name} 形状错误"
        assert np.all(np.isfinite(grad)), f"d{name} 包含非有限值"

    # 数值梯度检查：抽查几个参数位置，验证 BPTT 公式。
    checks = [
        ("Wxh", (0, inputs[0])),
        ("Whh", (1, 1)),
        ("Why", (targets[0], 0)),
        ("bh", (0, 0)),
        ("by", (targets[1], 0)),
    ]
    eps = 1e-5
    for name, idx in checks:
        params_pos = _copy_parameters(test_params)
        params_neg = _copy_parameters(test_params)
        params_pos[name][idx] += eps
        params_neg[name][idx] -= eps
        loss_pos = _loss_for_params(params_pos, inputs, targets, h0)
        loss_neg = _loss_for_params(params_neg, inputs, targets, h0)
        numeric = (loss_pos - loss_neg) / (2 * eps)
        analytic = grads["d" + name][idx]
        assert np.allclose(analytic, numeric, atol=1e-5, rtol=1e-4), (
            f"{name}{idx} 梯度检查失败: analytic={analytic}, numeric={numeric}"
        )

    print(f"✅ 练习 5 通过：BPTT 梯度形状、有限性和数值梯度检查均正确。loss = {loss:.4f}")

test_rnn_backward()

✅ 练习 5 通过：BPTT 梯度形状、有限性和数值梯度检查均正确。loss = 16.4786


## 6. 梯度裁剪与 `lossFun`

原始 `minimalRNN.py` 把前向传播、反向传播和梯度裁剪放在 `lossFun` 中：

```python
for dparam in [dWxh, dWhh, dWhy, dbh, dby]:
    np.clip(dparam, -5, 5, out=dparam)
return loss, dWxh, dWhh, dWhy, dbh, dby, hs[len(inputs)-1]
```

梯度裁剪的作用是把过大的梯度限制在固定范围内，缓解 RNN 训练中常见的梯度爆炸问题。

### 练习 6：实现梯度裁剪和 `lossFun`

为了后续训练代码更清晰，本实验让 `lossFun` 返回 `(loss, grads, hlast)`，其中 `grads` 是梯度字典。

In [13]:
### START CODE HERE ###
    # 原地裁剪每一个梯度数组。


    # 1. 前向传播，得到 loss、缓存和最后隐藏状态。

    # 2. 通过时间反向传播，得到梯度。

    # 3. 梯度裁剪。

### END CODE HERE ###

print("clip_gradients 与 lossFun 定义完成。")

clip_gradients 与 lossFun 定义完成。


In [14]:
# ===== 测试单元格（无需修改）=====
def test_lossFun_and_clipping():
    fake_grads = {
        "dWxh": np.array([[10.0, -8.0], [2.0, -3.0]]),
        "dWhh": np.array([[5.5, -5.5]]),
    }
    clipped = clip_gradients(fake_grads, -5, 5)
    assert clipped["dWxh"].max() <= 5 and clipped["dWxh"].min() >= -5, "dWxh 未正确裁剪"
    assert clipped["dWhh"].max() <= 5 and clipped["dWhh"].min() >= -5, "dWhh 未正确裁剪"

    inputs = [char_to_ix[ch] for ch in data[:seq_length]]
    targets = [char_to_ix[ch] for ch in data[1:seq_length + 1]]
    h0 = np.zeros((hidden_size, 1))
    loss, grads, hlast = lossFun(inputs, targets, h0, parameters)
    assert np.isfinite(loss), "lossFun 返回的 loss 应为有限数"
    assert hlast.shape == (hidden_size, 1), "lossFun 返回的 hlast 形状错误"
    for grad in grads.values():
        assert grad.max() <= 5 + 1e-12 and grad.min() >= -5 - 1e-12, "lossFun 应返回裁剪后的梯度"
    print("✅ 练习 6 通过：梯度裁剪与 lossFun 正确。")

test_lossFun_and_clipping()

✅ 练习 6 通过：梯度裁剪与 lossFun 正确。


## 7. 采样生成

训练时模型学习的是“给定当前字符和历史隐藏状态，预测下一个字符”。采样时也使用同一个递推过程：

1. 从一个种子字符 `seed_ix` 开始；
2. 计算隐藏状态和输出概率；
3. 按概率随机抽取下一个字符；
4. 把抽到的字符作为下一步输入，继续生成。

这就是 `minimalRNN.py` 中的 `sample(h, seed_ix, n)`。博客里的生成文本也是这个思想：每次采样一个字符，再把它喂回模型。

### 练习 7：实现 `sample`

补全采样函数。注意 `np.random.choice` 的 `p` 参数需要一维概率数组，因此要使用 `p.ravel()`。

In [15]:
def sample(h, seed_ix, n, parameters):
    Wxh, Whh, Why = parameters["Wxh"], parameters["Whh"], parameters["Why"]
    bh, by = parameters["bh"], parameters["by"]

    x = one_hot(seed_ix, vocab_size)
    ixes = []

    ### START CODE HERE ###
        # 1. 用当前输入 x 和上一隐藏状态 h 计算新隐藏状态。

        # 2. 计算 logits 和 softmax 概率。

        # 3. 按概率采样下一个字符 id。

        # 4. 把采样结果变成下一步输入。
    ### END CODE HERE ###

    return ixes

print("sample 定义完成。")

sample 定义完成。


In [16]:
# ===== 测试单元格（无需修改）=====
def test_sample():
    np.random.seed(7)
    h0 = np.zeros((hidden_size, 1))
    seed_ix = char_to_ix['a']
    ixes = sample(h0, seed_ix, 30, parameters)
    assert len(ixes) == 30, "sample 应生成指定长度的 id 列表"
    assert all(isinstance(ix, (int, np.integer)) for ix in ixes), "sample 的每个元素都应是整数 id"
    assert all(0 <= ix < vocab_size for ix in ixes), "sample 生成了词表范围外的 id"
    txt = ''.join(ix_to_char[ix] for ix in ixes)
    print("✅ 练习 7 通过：采样输出合法。")
    print("未训练模型的随机采样示例:")
    print(repr(txt))

test_sample()

✅ 练习 7 通过：采样输出合法。
未训练模型的随机采样示例:
'buksznmagmrujagxely\npyfnxcntrl'


## 8. 训练循环：滑动指针、平滑损失与 AdaGrad

现在把所有模块合起来。原始 `minimalRNN.py` 的训练循环包含以下关键变量：

| 变量 | 作用 |
|---|---|
| `n` | 迭代次数计数器 |
| `p` | 当前语料指针，按 `seq_length` 向右移动 |
| `hprev` | 上一段序列结束时的隐藏状态 |
| `smooth_loss` | 平滑后的损失，用于观察趋势 |
| `mWxh/mWhh/...` | AdaGrad 的平方梯度累积缓存 |

AdaGrad 更新公式是：

$$G \leftarrow G + g^2$$

$$\theta \leftarrow \theta - \eta \frac{g}{\sqrt{G + 10^{-8}}}$$

其中 $G$ 是历史平方梯度之和，$g$ 是当前梯度。

### 练习 8：实现有限步训练函数

补全 `train_rnn` 中与 `minimalRNN.py` 对应的核心训练逻辑：

1. 序列到达末尾或第一次迭代时，重置隐藏状态和数据指针；
2. 从 `data[p:p+seq_length]` 构造 `inputs`，从右移一位的片段构造 `targets`；
3. 定期调用 `sample` 观察模型输出；
4. 调用 `lossFun` 得到梯度；
5. 更新平滑损失；
6. 使用 AdaGrad 更新每个参数；
7. 移动数据指针并增加迭代计数。

In [17]:
def train_rnn(data, parameters, num_iters=400, sample_every=100, print_every=100):
    n, p = 0, 0
    hprev = np.zeros((hidden_size, 1))

    # AdaGrad memory variables，对应原始文件中的 mWxh/mWhh/mWhy/mbh/mby。
    mem = {name: np.zeros_like(value) for name, value in parameters.items()}

    smooth_loss = -np.log(1.0 / vocab_size) * seq_length
    smooth_history = []
    raw_history = []
    samples = []

    while n < num_iters:
        ### START CODE HERE ###
        # 1. 到达语料末尾，或第一次迭代时，重置隐藏状态与数据指针。

        # 2. 准备输入字符 id 和目标字符 id。

        # 3. 定期从当前模型采样，观察生成质量。

        # 4. 前向 + 反向 + 梯度裁剪。

        # 5. 平滑损失。原始代码使用 0.999，这里保留同样思想。

        # 6. AdaGrad 参数更新。

        # 7. 移动数据指针和迭代计数器。
        ### END CODE HERE ###

        smooth_history.append(float(smooth_loss))
        raw_history.append(float(loss))

        if print_every and n % print_every == 0:
            print(f"iter {n:4d}, raw loss: {loss:.4f}, smooth loss: {smooth_loss:.4f}")

    return parameters, smooth_history, raw_history, samples

print("train_rnn 定义完成。")

train_rnn 定义完成。


In [18]:
# ===== 测试单元格（无需修改）=====
def test_train_rnn_smoke():
    test_params = _copy_parameters(parameters)
    trained_params, smooth_history, raw_history, samples = train_rnn(
        data,
        test_params,
        num_iters=8,
        sample_every=4,
        print_every=0,
    )
    assert len(smooth_history) == 8, "smooth_history 长度应等于训练步数"
    assert len(raw_history) == 8, "raw_history 长度应等于训练步数"
    assert len(samples) == 2, "应在第 0 和第 4 次迭代各采样一次"
    assert all(np.isfinite(v) for v in smooth_history), "smooth_history 中不应有非有限值"
    assert all(np.isfinite(v) for v in raw_history), "raw_history 中不应有非有限值"
    changed = any(not np.allclose(trained_params[name], parameters[name]) for name in parameters)
    assert changed, "训练后参数应发生变化"
    print("✅ 练习 8 通过：训练循环、采样记录和参数更新均可运行。")

test_train_rnn_smoke()

✅ 练习 8 通过：训练循环、采样记录和参数更新均可运行。


## 9. 正式训练一个小模型

下面使用上面实现的完整训练循环训练 350 步。这个模型很小，语料也很小，目标不是生成高质量名字，而是确认整条 RNN 训练链路可以闭环运行。

In [19]:
# 重新初始化参数，避免前面的 smoke test 影响正式训练。
np.random.seed(1)
parameters = {
    "Wxh": np.random.randn(hidden_size, vocab_size) * 0.01,
    "Whh": np.random.randn(hidden_size, hidden_size) * 0.01,
    "Why": np.random.randn(vocab_size, hidden_size) * 0.01,
    "bh": np.zeros((hidden_size, 1)),
    "by": np.zeros((vocab_size, 1)),
}

trained_parameters, smooth_history, raw_history, training_samples = train_rnn(
    data,
    parameters,
    num_iters=350,
    sample_every=100,
    print_every=50,
)

print("\n训练过程中的采样片段：")
for step, txt in training_samples:
    compact = txt.replace('\n', '|')
    print(f"iter {step:4d}: {compact[:100]}")

iter   50, raw loss: 80.3622, smooth loss: 82.1713
iter  100, raw loss: 63.3400, smooth loss: 81.2170
iter  150, raw loss: 52.5228, smooth loss: 79.7850
iter  200, raw loss: 43.8330, smooth loss: 78.0387
iter  250, raw loss: 29.8239, smooth loss: 75.9665
iter  300, raw loss: 23.0163, smooth loss: 73.6589
iter  350, raw loss: 17.7195, smooth loss: 71.2170

训练过程中的采样片段：
iter    0: flcaudczarjdgswgikvtentlneqaijpyenupscvhkbrvmbbcmp|azglqsbbxhsslbxvklnjmlgwaitmumzw|lwiskbhhqyrhrbje
iter  100: lcileraniimw|elrh|frin|rerbujv|unrrelcosqmaybricoytlrcu|hulramiiknori|bula|doylr|wliiy|dara|hul|aqbr
iter  200: lcy|sun|ela|armanla|graen|juryl|enda|wa|grirera|o|fle|frirele|vy|menhin|kehe|aeri|qmanice|hen|reula|
iter  300: lly|fle|belannca|san|an|aeca|snel|keliy|gliel|adnen|tira|a|elny|harry|hulnneinbe||bon|arlla|aully|ju


In [20]:
# ===== 测试单元格（无需修改）=====
def test_full_training_result():
    assert len(smooth_history) == 350, "正式训练步数不正确"
    assert len(raw_history) == 350, "raw_history 长度不正确"
    assert len(training_samples) == 4, "应在 iter 0,100,200,300 采样"
    assert all(np.isfinite(v) for v in smooth_history), "smooth_history 中存在非有限值"
    assert all(np.isfinite(v) for v in raw_history), "raw_history 中存在非有限值"
    assert min(raw_history[-50:]) < raw_history[0], "训练后最近 50 步中应出现低于初始 loss 的结果"
    for name, value in trained_parameters.items():
        assert np.all(np.isfinite(value)), f"参数 {name} 出现非有限值"
    print("✅ 完整训练测试通过：loss 有下降迹象，参数保持数值稳定。")
    print(f"初始 raw loss: {raw_history[0]:.4f}")
    print(f"最近 50 步最低 raw loss: {min(raw_history[-50:]):.4f}")

test_full_training_result()

✅ 完整训练测试通过：loss 有下降迹象，参数保持数值稳定。
初始 raw loss: 82.3913
最近 50 步最低 raw loss: 17.7195


## 10. 使用训练后的模型生成字符

最后用训练后的参数采样几段文本。由于这个模型非常小、训练步数也少，输出不一定总是像真实名字；但如果训练链路正确，生成文本通常会逐渐出现换行、元音和名字片段等语料结构。

博客还讨论了 temperature：低温会让采样更保守，高温会让结果更多样但也更容易出错。本实验的基础 `sample` 沿用 `minimalRNN.py` 的原始写法，没有额外加入温度；你可以在选做题中自行扩展。

In [21]:
np.random.seed(2025)

print("--- 训练后模型采样 ---")
for i in range(10):
    h0 = np.zeros((hidden_size, 1))
    seed_ix = char_to_ix['a']
    generated_ix = sample(h0, seed_ix, 80, trained_parameters)
    generated = ''.join(ix_to_char[ix] for ix in generated_ix)
    print(f"sample {i + 1:02d}: {generated.replace(chr(10), '|')}")

--- 训练后模型采样 ---
sample 01: |quila|clmis|s|eny|movidor|sbelya|belly|ullilla|benna|beray|belvexra|len|fra|tan
sample 02: n|ale|aaei||nne|annie|ra|vlca|pbella|bil|donma|eiqon|elra|dor|mara|kictol|wellel
sample 03: |aara|frane|dory|siul|naleryy|hel|dorle|beliy|zmer|parirananie|arsiler|pdonca|vi
sample 04: ra|elie|benaa|doriweiranulena|enna|bicecyele|jlivan||bella|beray|lely|zoenna|hel
sample 05: llla|ilre||arap|unna|qulil|tara|malven|aenna|bennevqulla|lilara|a|ia|eileca|menn
sample 06: |arle|abel|sinn|panda|pavlen|emina|melly|cara|lanvenaranaa|grae|ela|a|ebeca|minn
sample 07: |lavar|wnel|banv|vare|ellemenaive|helvic|un|are|eriy|yeranben|belry|clityrfrari|
sample 08: |benl|dorice|jtoe|zoriny|hann|y|var|nnna|bella|ben|ben|bell|eby|a|rine|aray|pavi
sample 09: |anm|onele|abella|be|la|dera|coui|tylna|belaecara|beltar|envaer|yara|dorin||jonv
sample 10: |aamara|ewixy|bella|beca|dorkneriw|rvidon|jure|y|wa|hary|xavinj|be|annlen|benla|


In [22]:
# ===== 测试单元格（无需修改）=====
def test_generation_after_training():
    np.random.seed(11)
    h0 = np.zeros((hidden_size, 1))
    ixes = sample(h0, char_to_ix['a'], 60, trained_parameters)
    text = ''.join(ix_to_char[ix] for ix in ixes)
    assert len(text) == 60, "sample 应生成指定长度文本"
    assert all(ch in chars for ch in text), "生成文本包含词表外字符"
    assert any(ch in text for ch in 'aeiou\n'), "生成文本应包含语料中常见的元音或换行"
    print("✅ 训练后采样测试通过：生成文本长度、字符合法性正确。")
    print("测试采样:", repr(text[:80]))

test_generation_after_training()

✅ 训练后采样测试通过：生成文本长度、字符合法性正确。
测试采样: '\naara\n\nerry\nrica\na\nenna\nbenl\nellie\nztictar\nedimy\nracben\narin'


---

## 总结

本实验从 `minimalRNN.py` 出发，并参考 Karpathy 的 RNN 博客，把字符级 RNN 拆成了可测试的课堂练习。你现在应当能够解释以下模块在原始代码中的作用：

| 模块 | 关键知识点 |
|---|---|
| 字符词表 | `chars`、`char_to_ix`、`ix_to_char` |
| one-hot 输入 | 把字符 id 转为列向量 $x_t$ |
| RNN 前向传播 | $h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b_h)$ |
| 输出与损失 | logits、softmax、负对数似然 |
| BPTT | 从后往前累积 `dhnext` 与参数梯度 |
| 梯度裁剪 | 用 `np.clip` 缓解梯度爆炸 |
| 采样生成 | 用模型输出概率递推生成字符 |
| 训练循环 | 数据指针、隐藏状态续接、平滑 loss、AdaGrad |

**思考题（选做）：**

1. 如果把 `Whh` 置零，模型还能利用历史字符吗？为什么？
2. 如果去掉梯度裁剪，尝试增大学习率到 `1.0`，训练会发生什么？
3. `seq_length` 太短或太长分别会带来什么影响？
4. 本实验的 `softmax` 为什么要减去最大值？它是否改变了概率分布？
5. 为 `sample` 增加 temperature 参数。对比 `T=0.3`、`T=1.0`、`T=1.5` 的生成差异。
6. 尝试把 `hidden_size` 改为 100，与 `minimalRNN.py` 保持一致，观察训练速度和生成结果的变化。

参考资料：

- Andrej Karpathy, [The Unreasonable Effectiveness of Recurrent Neural Networks](https://karpathy.github.io/2015/05/21/rnn-effectiveness/)
- 本目录下的 `minimalRNN.py`